In [ ]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prasad0175/Credit-card-fraud-detection/blob/main/fraud_detection_ai.ipynb)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_score,
    recall_score
)

In [ ]:
# STEP 1: Upload CSV file manually
from google.colab import files

uploaded = files.upload()  # A button will appear — click it and select creditcard.csv
print("✓ File uploaded successfully")

In [ ]:
# STEP 2: Load Dataset
# --------------------------------------------------------------
print("Loading dataset...")
#df = pd.read_csv('C:\MBA\Problem_Solving_In_AI\creditcard.csv') #change it to your local directory
df = pd.read_csv('/content/creditcard.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nClass Distribution:")
print(df['Class'].value_counts())
print(f"\nFraud Percentage: {round(df['Class'].mean() * 100, 2)}%")

In [ ]:
# STEP 3: Exploratory Data Analysis (EDA)
# --------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df, palette=['steelblue', 'crimson'])
plt.title('Class Distribution: Legitimate vs Fraudulent Transactions')
plt.xlabel('Class (0 = Legitimate, 1 = Fraud)')
plt.ylabel('Number of Transactions')

fraud_pct      = round(df['Class'].mean() * 100, 2)
legitimate_pct = round(100 - fraud_pct, 2)
plt.xticks([0, 1], [f'Legitimate {legitimate_pct}%', f'Fraud {fraud_pct}%'])
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print("Figure saved: class_distribution.png")

In [ ]:
# STEP 4: Preprocessing
# --------------------------------------------------------------
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time']   = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df.drop(['Amount', 'Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

print(f"\nFeatures Shape: {X.shape}")
print(f"Target Shape  : {y.shape}")

In [ ]:
# STEP 5: Train/Test Split (80% Train, 20% Test)
# --------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set size : {X_train.shape[0]} samples")
print(f"Testing set size  : {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

In [ ]:
# STEP 6: Train All Three Models
# --------------------------------------------------------------

# ── Model 1: Isolation Forest (Unsupervised) ──────────────────
print("\n[1/3] Training Isolation Forest...")
iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train)
iso_pred_raw = iso_model.predict(X_test)
iso_pred     = np.where(iso_pred_raw == -1, 1, 0)
iso_score    = -iso_model.score_samples(X_test)
print("Isolation Forest training complete.")

In [ ]:
# ── Model 2: Logistic Regression (Supervised) ────────────────
print("\n[2/3] Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    
    #class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
lr_model.fit(X_train, y_train)

lr_pred  = lr_model.predict(X_test)
lr_score = lr_model.predict_proba(X_test)[:, 1]
print("Logistic Regression training complete.")

In [ ]:
# ── Model 3: Random Forest (Supervised) ──────────────────────
print("\n[3/3] Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    #class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred  = rf_model.predict(X_test)
rf_score = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest training complete.")

In [ ]:
# STEP 7: Evaluation — Classification Reports
# --------------------------------------------------------------
for name, y_pred in [
    ("Isolation Forest",    iso_pred),
    ("Logistic Regression", lr_pred),
    ("Random Forest",       rf_pred),
]:
    print("\n" + "="*55)
    print(f"CLASSIFICATION REPORT — {name}")
    print("="*55)
    print(classification_report(y_test, y_pred,
          target_names=['Legitimate', 'Fraud']))

In [ ]:
# STEP 8: Confusion Matrix Plots
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, y_pred) in zip(axes, [
    ("Isolation Forest",    iso_pred),
    ("Logistic Regression", lr_pred),
    ("Random Forest",       rf_pred),
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'])
    ax.set_title(f'Confusion Matrix\n{name}')
    ax.set_ylabel('Actual Class')
    ax.set_xlabel('Predicted Class')

plt.tight_layout()
plt.savefig('confusion_matrix_all.png', dpi=150)
plt.show()
print("Figure saved: confusion_matrix_all.png")

In [ ]:
# STEP 9: ROC Curve Comparison
# --------------------------------------------------------------
fpr_iso, tpr_iso, _ = roc_curve(y_test, iso_score)
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, lr_score)
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, rf_score)

auc_iso = roc_auc_score(y_test, iso_score)
auc_lr  = roc_auc_score(y_test, lr_score)
auc_rf  = roc_auc_score(y_test, rf_score)

plt.figure(figsize=(7, 5))
plt.plot(fpr_iso, tpr_iso, color='purple',    lw=2, label=f'Isolation Forest (AUC = {round(auc_iso, 4)})')
plt.plot(fpr_lr,  tpr_lr,  color='steelblue', lw=2, label=f'Logistic Regression (AUC = {round(auc_lr,  4)})')
plt.plot(fpr_rf,  tpr_rf,  color='darkorange',lw=2, label=f'Random Forest (AUC = {round(auc_rf,  4)})')
plt.plot([0, 1],  [0, 1],  color='navy', lw=1, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve Comparison — Fraud Detection')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()
print("Figure saved: roc_curve.png")

In [ ]:
# STEP 10: Feature Importance (Random Forest)
# --------------------------------------------------------------
feature_importances = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=feature_importances.values,
            y=feature_importances.index,
            palette='viridis')
plt.title('Top 10 Most Important Features - Random Forest')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print("Figure saved: feature_importance.png")

In [ ]:
# STEP 11: Final Summary Results
# --------------------------------------------------------------
summary = pd.DataFrame({
    'Model': ['Isolation Forest', 'Logistic Regression', 'Random Forest'],
    'Precision': [
        precision_score(y_test, iso_pred, zero_division=0),
        precision_score(y_test, lr_pred),
        precision_score(y_test, rf_pred),
    ],
    'Recall': [
        recall_score(y_test, iso_pred),
        recall_score(y_test, lr_pred),
        recall_score(y_test, rf_pred),
    ],
    'F1-Score': [
        f1_score(y_test, iso_pred),
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred),
    ],
    'ROC-AUC': [
        round(auc_iso, 4),
        round(auc_lr,  4),
        round(auc_rf,  4),
    ],
})

summary = summary.set_index('Model').round(4)

print("\n" + "="*55)
print("FINAL RESULTS SUMMARY")
print("="*55)
print(f"Preprocessing  : StandardScaler (no SMOTE)")
print(f"Imbalance      : class_weight='balanced' (LR & RF)")
print(f"Dataset        : Kaggle Credit Card Fraud")
print(f"Train/Test     : 80% / 20%")
print("="*55)
print(summary.to_string())
print("="*55)
print("\nAll figures saved. Code execution complete.")